In [78]:
from main import train_epoch
from data import get_data

special_symbols = {
            "pad": {"cont": [0.,1.],"CEL":-150},
            "bos": {"cont": [1.,1.], "CEL":-100},
            "eos": {"cont": [1.,0.],"CEL":-50},
            "sample": [0.,0.]
    }
batch_size = 5
dir_path = "/Users/paulwahlen/Desktop/Internship/ML/Code/TransfoV1/data"
vocab_charges, vocab_pdgs, E_label_RMSNormalizer, data_ld = get_data(dir_path, batch_size, special_symbols, "training")

ValueError: too many values to unpack (expected 2)

In [3]:
import torch
feats5,labels5 =  next(iter(data_ld))
feats5_reduced = feats5[:,::100]
print(feats5_reduced.size(dim = 1))
torch.set_printoptions(profile= "full")
print(feats5_reduced[0])

NameError: name 'data_ld' is not defined

In [3]:
def train_epoch(model, optim, train_dl, special_symbols,vocab_charges, vocab_pdgs, 
          hyperweights_lossfn, loss_fn_charges, loss_fn_pdg,loss_fn_cont):
    model.train() #setting model into train mode
    loss_epoch = 0.0
    for src,tgt in train_dl:
        src.to(DEVICE)
        tgt.to(DEVICE)

        src_padding_mask, tgt_padding_mask = create_mask(src,tgt,special_symbols["pad"]["cont"])
        tgt_in_padding_mask = tgt_padding_mask[:,:-1,:]
        tgt_out_padding_mask = tgt_padding_mask[:,1:,:]

        tgt_in = tgt[:,:-1] #sets the dimensions of transformer output -> must have the same as tgt_out
        logits_charges, logits_pdg, logits_cont = model(src,tgt_in, 
                                                                    src_padding_mask,
                                                                    tgt_in_padding_mask,
                                                                    src_padding_mask)
        optim.zero_grad()

        tgt_out = tgt[:,1:,:] #logits are compared with tokens shifted
        tgt_out_charges = tgt_out[...,0]
        tgt_out_pdg = tgt_out[...,1]
        tgt_out_cont = tgt_out[...,2:-2] #only (E, theta, phi)
        #Using spherical coordinates to get 3D direction vectors
        tgt_out_sin_theta = torch.sin(tgt_out_cont[...,0])
        tgt_out_cont[...,0] = torch.cos(tgt_out_cont[...,1]) * tgt_out_sin_theta
        tgt_out_cont[...,1] = torch.sin(tgt_out_cont[...,1]) * tgt_out_sin_theta
        tgt_out_cont = torch.concat([tgt_out_cont, torch.cos(tgt_out_cont[...,0]).unsqueeze_(-1)], dim = -1)
        #setting values of special tokens to 0 so that it doesn't contribute to loss
        tgt_out_cont[tgt_out_padding_mask] = torch.zeros(tgt_out_cont.size[-1])
        logits_cont[tgt_out_padding_mask] = torch.zeros(tgt_out_cont.size[-1])
        #eos and bos positions:
        eos_bos_mask = ((tgt_out_charges == vocab_charges.get_index(special_symbols["eos"]["CEL"])) 
                        or (tgt_out_charges == vocab_charges.get_index(special_symbols["bos"]["CEL"]))) 
        logits_cont[eos_bos_mask] = 0.0 #not sure if broadcasting works here
        
        #Computing the losses
        loss_charges = loss_fn_charges(logits_charges.transpose(dim0 = -2, dim1 = -1), tgt_out_charges)
        loss_pdg = loss_fn_pdg(logits_pdg.transpose(dim0 = -2, dim1 = -1), tgt_out_pdg)
        loss_cont_vec = loss_fn_cont(logits_cont, tgt_out_cont, reduction = 'none')
        npad = torch.count_nonzero(tgt_out_padding_mask, dim = -1)
        n_nopad = tgt_out_padding_mask.shape[-1] - npad
        loss_cont = torch.mean(loss_cont_vec * n_nopad)

        loss_vec = torch.tensor([loss_charges,loss_pdg, loss_cont])
        loss = torch.dot(torch.tensor(hyperweights_lossfn),loss_vec)

        loss.backward()
        optim.step()

        loss_epoch += loss.item()

    return loss_epoch / len(list(train_dl))

In [17]:
from data import create_mask
src_padding, tgt_padding = create_mask(feats5_reduced, labels5, special_symbols["pad"]["cont"])
print(feats5_reduced[~src_padding].shape)
print(src_padding.shape)

torch.Size([74, 6])
torch.Size([5, 23])


In [14]:
#print(vocab_charges.vocab)
#print(vocab_charges.vocab["-50"])
#print(vocab_charges.vocab.keys())
keys = list(vocab_charges.vocab.keys())
eos = keys[0]
print(eos)
print(vocab_charges.get_index(eos))
print(vocab_charges.get_index(special_symbols["eos"]["CEL"]))
print(vocab_charges.get_index(special_symbols["bos"]["CEL"]))
eos_bos_mask = ((labels5[...,0] == vocab_charges.get_index(special_symbols["eos"]["CEL"])) 
                        + (labels5[...,0] == vocab_charges.get_index(special_symbols["bos"]["CEL"]))) 
print(eos_bos_mask[0])
print(labels5[0])

-150
tensor(0)
tensor(2)
tensor(1)
tensor([ True, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False,  True, False, False, False, False, False, False, False, False,
        False, False])
tensor([[ 1.0000,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0

In [13]:
bool1 = torch.tensor([True, True, False, False])
bool2 = torch.tensor([False, True, False, True])
print(bool1+bool2)

tensor([ True,  True, False,  True])


In [16]:
labels5[eos_bos_mask] = 0.
print(labels5[0])

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0.1540,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4538,  0.3886,  0.6474,  0.6557,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4539,  0.5872,  0.7305,  0.3486,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4470,  0.8047,  0.2092,  0.5556,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4539, -0.7112, -0.1417,  0.6885,  0.0000,  0.0000],
        [ 3.0000

In [19]:
special_mask = tgt_padding + eos_bos_mask
print(special_mask[0])
labels5[special_mask] = 0.
print(labels5[0])

tensor([ True, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True])
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0.1540,  0.0000,  0.0000],
        [

In [26]:
nspe_tokens = torch.count_nonzero(special_mask, dim = -1)
n_nospe = special_mask.shape[-1] - nspe_tokens
print(n_nospe)
print(torch.sum(n_nospe))
print(labels5[~special_mask].shape)

tensor([30, 23, 29, 18, 19])
tensor(119)
torch.Size([119, 8])


In [30]:
logits_cont = torch.rand((5,30,3)).fill_(1)
logits_cont_3d = torch.concat([logits_cont, torch.cos(logits_cont[...,2]).unsqueeze(-1)], dim = -1)
print(logits_cont_3d)


tensor([[[1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1

In [37]:
logits_charges = torch.rand(batch_size,30,6)
truth = torch.randint(0,6,(batch_size, 30))

print(logits_charges)
print(truth)

tensor([[[8.9761e-01, 2.5491e-02, 3.6489e-01, 3.8359e-02, 4.1589e-01,
          9.5377e-01],
         [8.4005e-01, 4.8023e-01, 7.5554e-01, 1.6927e-01, 3.5464e-02,
          4.3823e-01],
         [2.0006e-01, 1.6610e-01, 3.6585e-01, 3.3457e-01, 1.2604e-01,
          8.8230e-01],
         [4.7979e-01, 3.8399e-01, 7.9198e-01, 3.3886e-01, 5.9310e-01,
          8.9021e-01],
         [4.4126e-01, 3.4168e-02, 5.6999e-01, 7.2213e-01, 9.8492e-01,
          5.9495e-01],
         [8.3847e-01, 6.0544e-01, 9.7585e-01, 3.3388e-01, 9.4004e-01,
          9.9251e-01],
         [5.4694e-01, 1.1411e-01, 1.5854e-01, 7.4086e-01, 5.4851e-01,
          5.6418e-02],
         [2.3795e-02, 3.4105e-01, 1.0692e-02, 6.5822e-01, 1.8979e-01,
          9.9075e-01],
         [2.2163e-01, 4.1981e-01, 5.7117e-02, 1.8220e-01, 6.3997e-01,
          7.5596e-01],
         [9.0165e-01, 4.1330e-01, 7.9380e-01, 2.9616e-03, 5.8627e-01,
          5.1612e-02],
         [8.6417e-01, 2.0546e-01, 1.6061e-01, 8.9938e-01, 2.2077e-01,


In [41]:
import torch.nn as nn
Cross = nn.CrossEntropyLoss(ignore_index= 0)
loss = Cross(logits_charges.transpose(dim0 = -2, dim1 = -1), truth)
print(loss)

tensor(1.8420)


In [49]:
bos = torch.tensor([1,2,3])
clusters_transfo = torch.tile(bos, (batch_size,1))
print(clusters_transfo)

tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])


In [53]:
zeros = torch.zeros(batch_size,1).type(torch.bool)
zeros

tensor([[False],
        [False],
        [False],
        [False],
        [False]])

In [56]:
bool1 = torch.tensor([False,False,True,True])
bool2 = torch.tensor([True,False,True,False])
bool1 * bool2

tensor([False, False,  True, False])

In [63]:
next_charges_batch = torch.argmax(logits_charges[:,-1], dim = -1)
print(next_charges_batch.shape)
print(next_charges_batch)
#print(logits_charges)
is_done_prev = torch.zeros(batch_size).type(torch.bool)
new_eos_tokens_batch = ( (next_charges_batch == special_symbols["eos"]["CEL"]) * ~is_done_prev)
print(new_eos_tokens_batch)

torch.Size([5])
tensor([2, 4, 3, 1, 0])
tensor([False, False, False, False, False])


In [84]:
niter = 10
is_done_prev = torch.zeros(batch_size).type(torch.bool)
is_done = torch.zeros(batch_size).type(torch.bool)
tgt_key_padding =  torch.zeros(batch_size,1).type(torch.bool)
print(id(is_done) == id(is_done_prev))
logits_tot = torch.tile(torch.tensor([1]), (batch_size,1))
print(vocab_charges.get_index(special_symbols["eos"]["CEL"]))
for i in range(niter):
    logits_next_iter = torch.rand((batch_size,30,6))
    next_charges_batch = torch.argmax(logits_next_iter[:,-1], dim =-1)
    new_eos_tokens_batch = ( (next_charges_batch == vocab_charges.get_index(special_symbols["eos"]["CEL"])) * ~is_done_prev)
    print(new_eos_tokens_batch)
    if torch.count_nonzero(new_eos_tokens_batch) > 0:
        is_done[is_done_prev == False] = new_eos_tokens_batch[is_done_prev == False]
    next_charges_batch[is_done_prev] =0
    logits_tot = torch.concat([logits_tot, next_charges_batch.unsqueeze(-1)], dim = -1)
    tgt_key_padding = torch.hstack([tgt_key_padding, is_done_prev.unsqueeze(-1)])
    print("logits_tot ===================")
    print(logits_tot)
    print("==============================")
    is_done_prev = torch.clone(is_done)
    print("tgt_key_padding ==============")
    print(tgt_key_padding)
    print("==============================")
    
print(vocab_charges.indices_to_tokens(logits_tot))



False
tensor(2)
tensor([False, False,  True,  True,  True])
logits_tot ===================
tensor([[1, 3],
        [1, 3],
        [1, 2],
        [1, 2],
        [1, 2]])
tgt_key_padding ==============
tensor([[False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False]])
tensor([False, False, False, False, False])
logits_tot ===================
tensor([[1, 3, 3],
        [1, 3, 1],
        [1, 2, 0],
        [1, 2, 0],
        [1, 2, 0]])
tgt_key_padding ==============
tensor([[False, False, False],
        [False, False, False],
        [False, False,  True],
        [False, False,  True],
        [False, False,  True]])
tensor([ True,  True, False, False, False])
logits_tot ===================
tensor([[1, 3, 3, 2],
        [1, 3, 1, 2],
        [1, 2, 0, 0],
        [1, 2, 0, 0],
        [1, 2, 0, 0]])
tgt_key_padding ==============
tensor([[False, False, False, False],
        [False, False, False, False],
        [False, False,  T

AttributeError: 'Tensor' object has no attribute 'astype'

In [88]:
dict1 = {"one":1,
        "two":2,
        "three":3}
print(len(dict1.keys()))
torch.save(dict1, "dict1.pt")
dict1_from_file = torch.load("dict1.pt")
print(type(dict1_from_file))

3
<class 'dict'>


In [24]:
import torch.nn as nn
loss = nn.MSELoss()
input = torch.randn(3, 5, requires_grad=True)
target = torch.randn(3, 5)
output = loss(input, target)
output.backward(retain_graph=True)
first_grad = input.grad
print(input.grad)
input.grad.zero_
print(input.grad)
loss_vec = output + output
#loss_vec = torch.dot(torch.tensor([1.,1.]), loss_vec)
loss_vec.backward()
print(input.grad- first_grad)


tensor([[ 0.0977, -0.1512, -0.1803, -0.2301,  0.1387],
        [-0.0774,  0.1968, -0.2973, -0.2176,  0.1116],
        [-0.1250,  0.0392, -0.0475,  0.0377,  0.1280]])
tensor([[ 0.0977, -0.1512, -0.1803, -0.2301,  0.1387],
        [-0.0774,  0.1968, -0.2973, -0.2176,  0.1116],
        [-0.1250,  0.0392, -0.0475,  0.0377,  0.1280]])
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [31]:
import torch
def add_pad(input, pad_tokens, size):
    nhits_input = input.shape[1]
    diff_size = size - nhits_input
    if diff_size > 0:
        pad = pad_tokens.repeat(input.shape[0], diff_size,1)
        print(pad)
        input = torch.cat([input, pad], dim = 1)
    return input

a = torch.randint(5,(3,4,5))
print(a)
pad_tokens = torch.zeros(5)
a = add_pad(a,pad_tokens,7)
print(a)
b = torch.randint(5,(5,))
pad = [100,100]
print(b)
b.put_(torch.tensor([0,1], dtype = torch.long), torch.tensor(pad,dtype = torch.long).repeat(2,1))
print(b)


tensor([[[2, 1, 4, 0, 1],
         [1, 4, 2, 1, 3],
         [3, 0, 4, 1, 3],
         [4, 2, 1, 0, 3]],

        [[3, 4, 3, 1, 2],
         [1, 4, 3, 4, 3],
         [3, 4, 1, 2, 1],
         [4, 2, 0, 3, 1]],

        [[1, 0, 4, 3, 2],
         [2, 3, 0, 2, 1],
         [1, 4, 4, 1, 1],
         [0, 1, 1, 0, 4]]])
tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]])
tensor([[[2., 1., 4., 0., 1.],
         [1., 4., 2., 1., 3.],
         [3., 0., 4., 1., 3.],
         [4., 2., 1., 0., 3.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[3., 4., 3., 1., 2.],
         [1., 4., 3., 4., 3.],
         [3., 4., 1., 2., 1.],
         [4., 2., 0., 3., 1.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0

IndexError: put_(): Expected source and index to have the same number of elements, but got source.numel() = 4, index.numel() = 2

In [37]:
mask = torch.randint(2, (50,1)).type(torch.bool)
values = torch.rand(50,4)
print(values)
print(mask)
print(values[mask])

tensor([[0.7890, 0.1331, 0.4234, 0.8666],
        [0.3149, 0.1456, 0.4216, 0.7878],
        [0.9233, 0.2728, 0.8791, 0.0083],
        [0.7062, 0.7766, 0.3038, 0.4634],
        [0.6554, 0.8491, 0.9581, 0.8218],
        [0.6232, 0.7234, 0.9301, 0.3972],
        [0.7761, 0.4018, 0.0480, 0.8590],
        [0.7104, 0.9474, 0.4553, 0.0183],
        [0.6262, 0.8567, 0.6337, 0.8935],
        [0.7989, 0.4646, 0.0220, 0.2751],
        [0.2697, 0.9762, 0.6586, 0.4370],
        [0.4601, 0.7807, 0.5870, 0.3410],
        [0.2031, 0.1452, 0.0875, 0.7305],
        [0.4438, 0.1419, 0.1465, 0.6423],
        [0.0854, 0.1246, 0.9943, 0.1900],
        [0.4587, 0.0122, 0.3164, 0.2670],
        [0.1732, 0.0800, 0.0920, 0.6570],
        [0.7932, 0.8225, 0.9308, 0.6531],
        [0.0749, 0.2988, 0.4941, 0.9603],
        [0.6030, 0.9501, 0.7794, 0.8441],
        [0.4434, 0.5237, 0.4783, 0.3001],
        [0.1916, 0.3966, 0.9273, 0.0398],
        [0.8019, 0.7731, 0.3862, 0.1561],
        [0.2561, 0.4356, 0.5590, 0

IndexError: The shape of the mask [50, 1] at index 1 does not match the shape of the indexed tensor [50, 4] at index 1

In [45]:
next_DOF = torch.rand(50,4)
special_symbols = torch.tensor([0,1])
print(special_symbols.unsqueeze(0))
next_DOF = torch.cat((next_DOF, special_symbols.repeat(50,1)), dim = -1)
print(next_DOF)

tensor([[0, 1]])
tensor([[0.7910, 0.7597, 0.9612, 0.6086, 0.0000, 1.0000],
        [0.9667, 0.4430, 0.1017, 0.7463, 0.0000, 1.0000],
        [0.2534, 0.3435, 0.2534, 0.4476, 0.0000, 1.0000],
        [0.4197, 0.3102, 0.0207, 0.2897, 0.0000, 1.0000],
        [0.7925, 0.6399, 0.5210, 0.0379, 0.0000, 1.0000],
        [0.5381, 0.8017, 0.8273, 0.1425, 0.0000, 1.0000],
        [0.4935, 0.8802, 0.6868, 0.7600, 0.0000, 1.0000],
        [0.6421, 0.6177, 0.6823, 0.3069, 0.0000, 1.0000],
        [0.5256, 0.0837, 0.7296, 0.0644, 0.0000, 1.0000],
        [0.9528, 0.7999, 0.4424, 0.5145, 0.0000, 1.0000],
        [0.0766, 0.8029, 0.2066, 0.0420, 0.0000, 1.0000],
        [0.6832, 0.1915, 0.6002, 0.7772, 0.0000, 1.0000],
        [0.8472, 0.8019, 0.2174, 0.2653, 0.0000, 1.0000],
        [0.3182, 0.0617, 0.8685, 0.3994, 0.0000, 1.0000],
        [0.3113, 0.1166, 0.6946, 0.7603, 0.0000, 1.0000],
        [0.3821, 0.4271, 0.3201, 0.2649, 0.0000, 1.0000],
        [0.0093, 0.0659, 0.9332, 0.0303, 0.0000, 1.0000

In [47]:
mask = [False, True]
a = torch.tensor([4,5])
a[mask] = 0
print(a)

tensor([4, 0])


In [56]:
batch_size = 10
is_done = torch.zeros(batch_size,1).type(torch.bool)
is_done_prev = torch.randint(2,(batch_size,1)).type(torch.bool)
print(is_done)
print(is_done_prev == False)
print(~is_done_prev)
print(is_done[~is_done_prev])
is_done_prev
mask = torch.randint(2, (batch_size,)).type(torch.bool)
print(mask[~is_done_prev.squeeze(-1)])


tensor([[False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False]])
tensor([[ True],
        [False],
        [False],
        [ True],
        [False],
        [False],
        [False],
        [ True],
        [False],
        [ True]])
tensor([[ True],
        [False],
        [False],
        [ True],
        [False],
        [False],
        [False],
        [ True],
        [False],
        [ True]])
tensor([False, False, False, False])
tensor([False, False, False,  True])


In [59]:
import numpy as np
logits_cont = torch.rand(50,250,3)
logits_cont_sin_theta = torch.sin(logits_cont[...,-2]) #logits: (E, theta, phi)
logits_cont_nx = torch.cos(logits_cont[...,-1]) * logits_cont_sin_theta
logits_cont_ny = torch.sin(logits_cont[...,2]) * logits_cont_sin_theta
logits_cont = torch.concat([logits_cont[...,0].unsqueeze(-1),logits_cont_nx.unsqueeze(-1),logits_cont_ny.unsqueeze(-1),  torch.cos(logits_cont[...,1]).unsqueeze(-1)], dim = -1)
print(torch.sum(torch.square(logits_cont[...,1:4]), dim = -1))

tensor([[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        ...,
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000]])


In [60]:
next_DOF_cont_batch = torch.rand(50,3)
next_DOF_cont_batch_sin_theta = torch.sin(next_DOF_cont_batch[...,1])
next_DOF_cont_batch[...,0] = torch.cos(next_DOF_cont_batch[...,-2]) * next_DOF_cont_batch_sin_theta
next_DOF_cont_batch[...,1] = torch.sin(next_DOF_cont_batch[...,-2]) * next_DOF_cont_batch_sin_theta
next_DOF_cont_batch = torch.concat([next_DOF_cont_batch, torch.cos(next_DOF_cont_batch[...,0]).unsqueeze_(-1)], dim = -1)
print(torch.sum(torch.square(next_DOF_cont_batch[...,1:4]), dim = -1))

tensor([1.1707, 1.3088, 1.2514, 1.3175, 1.5388, 0.9993, 1.4564, 1.8125, 1.0852,
        1.6777, 0.8621, 1.0386, 1.7776, 0.7879, 1.3298, 0.8429, 0.9927, 1.0834,
        1.5998, 1.4488, 1.2665, 1.4104, 0.9486, 1.0087, 1.5015, 0.8328, 1.1133,
        1.6827, 1.6238, 1.0343, 0.7857, 1.2541, 1.6887, 1.1366, 0.9381, 1.0096,
        1.7477, 1.4769, 1.0056, 1.1667, 1.0960, 1.6606, 1.0850, 0.9456, 0.7101,
        0.8234, 1.2844, 1.1250, 1.5887, 1.6664])


In [62]:
def get_cartesian_from_angles(input):
    #logits: (E, theta, phi)
    theta = input[...,-2]
    phi = input[...,-1]
    sin_theta = torch.sin(theta) 
    nx = torch.cos(phi) * sin_theta
    ny = torch.sin(phi) * sin_theta
    return torch.concat([input[...,0].unsqueeze(-1),nx.unsqueeze(-1),ny.unsqueeze(-1), torch.cos(theta).unsqueeze(-1)], dim = -1)


In [63]:
print(torch.sum(torch.square(get_cartesian_from_angles(next_DOF_cont_batch)[...,1:]), dim = -1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [77]:
import awkward as ak  
a = ak.Array([[1,2,3], [2,4]])
len_ = ak.Array([3,1])
index = ak.Array([[True, True], [True]])
a[~index]


<Array [[], []] type='2 * var * int64'>

In [ ]:
def cartesian_to_angles(input):
    theta = np.arccos(input[...,-3])
    phi = np.arctan2(input[...,-4], input[...,-5])
    return theta, phi

def get_cartesian_from_angles(input):
    #logits: (E, theta, phi)
    theta = input[...,-2]
    phi = input[...,-1]
    sin_theta = torch.sin(theta) 
    nx = torch.cos(phi) * sin_theta
    ny = torch.sin(phi) * sin_theta
    return torch.concat([input[...,0].unsqueeze(-1),
                         nx.unsqueeze(-1),
                         ny.unsqueeze(-1), 
                         torch.cos(theta).unsqueeze(-1)], dim = -1)

In [4]:
from torch.nn.modules.transformer import _get_clones
import torch.nn as nn
import torch
module = nn.Module()
clones = _get_clones(module,4)
print(clones.modules())
a = torch.rand((3,10,5))
print(a)
zerosa = torch.zeros(3,*a.shape)
ones = torch.ones_like(a)
zeros = torch.ones_like(a) * 5
zerosa[0] = a
zerosa[1] = ones
zerosa[2] = zeros
print(zerosa.shape)
out = torch.concatenate(zerosa, dim = -1)
print(out.shape)

#print(zerosa)

<generator object Module.modules at 0x7fa57a603b30>
tensor([[[0.8052, 0.2589, 0.9965, 0.5325, 0.6125],
         [0.5488, 0.2388, 0.7372, 0.7795, 0.5766],
         [0.8338, 0.2808, 0.5810, 0.6237, 0.4189],
         [0.2032, 0.6699, 0.3132, 0.6603, 0.5516],
         [0.2717, 0.2958, 0.8775, 0.6797, 0.2021],
         [0.8050, 0.1296, 0.6393, 0.2387, 0.6327],
         [0.1679, 0.1897, 0.0262, 0.2107, 0.0117],
         [0.4799, 0.7064, 0.7327, 0.5517, 0.9372],
         [0.6989, 0.1518, 0.7228, 0.3332, 0.9017],
         [0.8782, 0.9514, 0.5243, 0.8857, 0.9145]],

        [[0.0065, 0.0353, 0.0508, 0.9613, 0.5607],
         [0.1375, 0.2788, 0.7715, 0.3593, 0.1574],
         [0.8161, 0.9685, 0.9023, 0.1727, 0.9607],
         [0.2460, 0.6880, 0.0393, 0.3740, 0.4373],
         [0.8832, 0.5244, 0.5960, 0.9843, 0.7559],
         [0.8711, 0.3888, 0.6524, 0.8913, 0.8691],
         [0.6545, 0.9183, 0.5068, 0.7595, 0.1562],
         [0.5967, 0.7407, 0.1800, 0.9631, 0.4591],
         [0.3797, 0.0862, 0.

TypeError: concatenate() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim, *, Tensor out)
 * (tuple of Tensors tensors, name dim, *, Tensor out)


In [5]:
import torch
a = torch.rand((3,10,5))
zerosa = torch.zeros(3,*a.shape)
ones = torch.ones_like(a)
zeros = torch.ones_like(a) * 5
zerosa[0] = a
zerosa[1] = ones
zerosa[2] = zeros
print(zerosa.shape)
out = torch.concatenate(zerosa.chunk(3,dim = 0), dim = -1)
print(out)

torch.Size([3, 3, 10, 5])


TypeError: concatenate() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim, *, Tensor out)
 * (tuple of Tensors tensors, name dim, *, Tensor out)
